In [22]:
import numpy as np
import pandas as pd
import paretoset
import time

PATH = '/Users/shivshekhar/Desktop/Arka/bao'
MAX_NUM_SEQ_ALGO = 50
MAX_N_BOXES = 30
DIM_FACTOR = 139

def CardStack(df):
    dims = [df['l'].max(), df['w'].max(), df['h'].sum()]
    dims.sort(reverse = True)
    return pd.DataFrame([dims], columns = ['l', 'w', 'h'])

def TwoItem(df):
    a = list(df.iloc[0])
    b = list(df.iloc[1])
    dims = []
    for i in range(3):
        for j in range(3):
            ac = a.copy()
            ae = ac.pop(i)
            bc = b.copy()
            be = bc.pop(j)
            dim = [ae + be, max(ac[0], bc[0]), max(ac[1], bc[1])]
            dim.sort(reverse = True)
            dims.append(dim)
    dims = pd.DataFrame(dims, columns = ['l', 'w', 'h'])
    return dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)

def TwoItemSeq(df_in, k_max = 9):
    df = df_in.copy()
    df['v'] = df['l'] * df['w'] * df['h']
    df = df.sort_values('v')[['l', 'w', 'h']]
    dims = df.iloc[[0]]
    for r in range(1, len(df.index)):
        # try pairwise addition of the new item to current superitems
        row = df.iloc[[r]]
        dims = pd.concat([TwoItem(pd.concat([row, dims.loc[[rd]]])) for rd in dims.index])
        dims = dims[paretoset.paretoset(dims, ['min', 'min', 'min'])]
        # reduce the number of superitems
        dims['v'] = dims['l'] * dims['w'] * dims['h']
        dims = dims.sort_values('v')[['l', 'w', 'h']].iloc[0:k_max].reset_index(drop = True)
    return dims

def CheckFit(dims, boxes):
    fit = pd.merge(boxes.assign(key = 1), dims.assign(key = 1), on = 'key').drop('key', axis = 1)
    fit['f'] = (fit['L'] >= fit['l']) & (fit['W'] >= fit['w']) & (fit['H'] >= fit['h'])
    fit = fit[['n', 'f']].groupby('n').max().reset_index()
    return fit


In [23]:

es = pd.read_csv('ryder_data.csv')
# es[['Order No', 'Shipment Cost']].drop_duplicates().sum()

# es[['Order No', 'Shipment Cost', 'Shipment Weight (lbs)', 'Shipment Dim. 1 (in)', 'Shipment Dim. 2 (in)', 'Shipment Dim. 3 (in)']].drop_duplicates()
# es[['Order No']].drop_duplicates()

# es['W_s'] = es['Shipment Weight (lbs)']
# es['W_i'] = es['line Item Weight (lbs)'] * es['line Item Quantity']
# es['V_i'] = es['Line Item Length (in)'] * es['Line Item Width (in)'] * es['Line Item Height (in)'] * es['line Item Quantity']
# es['V_s'] = es['Shipment Dim. 1 (in)'] * es['Shipment Dim. 2 (in)'] * es['Shipment Dim. 3 (in)']

# es_agg = es[['Order No', 'Shipment Cost', 'W_s', 'V_s', 'W_i', 'V_i']].fillna(0)
# es_agg = es_agg.groupby(['Order No', 'Shipment Cost', 'W_s', 'V_s'], as_index = False).sum()
# es_agg[['W_s', 'V_s', 'W_i', 'V_i']] = es_agg[['W_s', 'V_s', 'W_i', 'V_i']].fillna(0)
# es_agg['BW'] = es_agg.apply(lambda x: np.ceil(max(x['W_s'], x['W_i'], x['V_s'] / DIM_FACTOR, x['V_i'] / DIM_FACTOR)), axis = 1)
# t = es_agg[es_agg['Shipment Cost'] > 0][['Order No', 'Shipment Cost', 'BW']]
# # display(t)

# y = 'Shipment Cost'
# x = 'BW'
# n = len(t.index)

# x_m = t[x].mean()
# y_m = t[y].mean()

# SLOPE = ((t[x] - x_m) * (t[y] - y_m)).sum() / ((t[x] - x_m) * (t[x] - x_m)).sum()
# INTERCEPT = y_m - SLOPE * x_m

# print('slope and intercept:', round(SLOPE, 2), round(INTERCEPT, 2), 
#       'total cost:', t[y].sum(), 'total est cost:', round((t[x] * SLOPE + INTERCEPT).sum(), 2))

In [ ]:
y_m - 0.3 * x_m

In [20]:
FULFILLMENT_COST = lambda x: x['c'] + 5.15 + 0.3 * np.ceil(max(x['p'] + x['P'], x['V'] / DIM_FACTOR))
# FULFILLMENT_COST = lambda x: x['c'] + INTERCEPT + SLOPE * np.ceil(max(x['p'] + x['P'], x['V'] / DIM_FACTOR))

In [24]:
es_df = pd.read_csv('ryder_data.csv')
dim_col_names = ['Line Item Length (in)', 'Line Item Width (in)', 'Line Item Height (in)']
es_df = es_df[['Order No', 'line Item Quantity', 'line Item Weight (lbs)'] + dim_col_names]
es_df = es_df.rename(columns = {'Order No' : 'o', 'line Item Quantity' : 'q', 'line Item Weight (lbs)' : 'p'})
es_df['l'] = es_df[dim_col_names].max(axis = 1).astype(float)
es_df['h'] = es_df[dim_col_names].min(axis = 1).astype(float)
es_df['w'] = (es_df[dim_col_names].sum(axis = 1) - es_df['l'] - es_df['h']).astype(float)
es_df = es_df[['o', 'l', 'w', 'h', 'q', 'p']]
es_df

,o,l,w,h,q,p
0,D288-999-999,8.66,6.10,2.36,700,0.95
1,D286-999-999,6.10,3.54,1.38,690,1.15
2,P1043-999-999,16.14,9.25,8.27,685,0.31
3,D284-999-999,12.80,9.06,2.77,426,7.00
4,D285-999-999,6.50,4.53,1.97,323,1.28
...,...,...,...,...,...,...
9334,HL1-D16-055,12.20,9.45,4.72,1,3.28
9335,E4T-090-045,11.42,7.48,4.33,1,1.54
9336,268-001-090,12.60,8.66,4.72,1,1.64
9337,279-090-110,12.99,8.66,5.31,1,2.01


In [25]:
ul_bx = pd.read_csv('Uline_CSV_boxes.csv')

ul_bx['n'] = ul_bx['Title']
ul_bx['P'] = 0
ul_bx['l'] = ul_bx['Option1 Value'].apply(lambda x: float(x.split('x')[0]))
ul_bx['w'] = ul_bx['Option1 Value'].apply(lambda x: float(x.split('x')[1]))
ul_bx['h'] = ul_bx['Option1 Value'].apply(lambda x: float(x.split('x')[2]))
ul_bx['L'] = ul_bx[['l', 'w', 'h']].max(axis = 1).astype(float)
ul_bx['H'] = ul_bx[['l', 'w', 'h']].min(axis = 1).astype(float)
ul_bx['W'] = (ul_bx[['l', 'w', 'h']].sum(axis = 1) - ul_bx['l'] - ul_bx['h']).astype(float)
ul_bx['c'] = ul_bx['Variant Price']
ul_bx = ul_bx[ul_bx['c'] < 10]
ul_bx = ul_bx.sort_values('c').groupby(['L', 'W', 'H'], as_index = False).first()
ul_bx = ul_bx[['n', 'L', 'W', 'H', 'P', 'c']]
ul_bx

FileNotFoundError: [Errno 2] No such file or directory: '/Users/shivshekhar/Desktop/Arka/baoUline_CSV_boxes.csv'

In [7]:


# df = pd.DataFrame([[1234, 3, 2, 1, 3, 0.5], [1235, 3, 2, 1, 4, 0.5]], columns = ['o', 'l', 'w', 'h', 'q', 'p'])
df = es_df.copy()
orders = df['o'].drop_duplicates()
df = df.set_index('o')

# bx = pd.DataFrame([['A', 1, 1, 1, 0.1, 0.32], ['B', 10, 10, 10, 2.1, 1.54]], columns = ['n', 'L', 'W', 'H', 'P', 'c'])
bx = ul_bx.copy()

start = time.time()
iterator = 0
costs = []
for o in orders:
    odf = df.loc[[o]].reset_index(drop = True)
    i_weight = (odf['q'] * odf['p']).sum()
    i_num = odf['q'].sum()
    odf = pd.concat([pd.concat([odf[odf['q'] == q]] * q) for q in odf['q'].drop_duplicates()])[['l', 'w', 'h']]
    if i_num < MAX_NUM_SEQ_ALGO:
        dims = TwoItemSeq(odf)
    else:
        dims = CardStack(odf)
    # display(dims)
    
    cost = bx.copy().set_index('n')
    cost['V'] = cost['L'] * cost['W'] * cost['H']
    cost['p'] = i_weight
    cost['o'] = o
    cost['f'] = CheckFit(dims, bx[['n', 'L', 'W', 'H']]).set_index('n')['f']
    cost['t'] = cost.apply(FULFILLMENT_COST, axis = 1) * cost['f'].apply(lambda x: 1 if x else None)
    costs.append(cost[['o', 't']].reset_index().pivot(index = 'o', columns = 'n', values = 't'))
    
    # iterator += 1
    # if iterator == 1000:
    #     print(time.time() - start)
    #     break

print(time.time() - start)
costs = pd.concat(costs, axis = 0)
display(costs)
costs.to_csv(PATH + 'costs.csv')

624.6287879943848


n,S-10655,S-10656,S-10659,S-10661,S-10662,S-10666,S-11252,S-11368,S-11369,S-11371,...,S-4988,S-4989,S-4990,S-4991,S-4992,S-4993,S-4994,S-4997,S-521,S-559
o,,,,,,,,,,,,,,,,,,,,,
359104.0,15.97,28.17,17.99,35.75,24.36,20.21,8.69,9.04,NaN,8.61,...,8.85,11.99,8.29,9.04,8.36,10.37,9.83,19.17,9.4,14.81
359813.0,15.97,28.17,17.99,35.75,24.36,20.21,8.39,9.04,NaN,8.61,...,8.85,11.99,7.09,9.04,7.76,10.37,9.83,19.17,9.4,14.81
361807.0,NaN,NaN,NaN,NaN,24.36,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
362411.0,NaN,NaN,NaN,NaN,24.36,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
362461.0,NaN,NaN,NaN,NaN,24.36,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377357.0,15.97,28.17,17.99,35.75,24.36,20.21,8.39,9.04,NaN,8.61,...,8.85,11.99,7.09,9.04,7.76,10.37,9.83,19.17,9.4,14.81
377358.0,NaN,NaN,NaN,NaN,NaN,20.21,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
377359.0,15.97,28.17,17.99,35.75,24.36,20.21,8.39,9.04,NaN,8.61,...,8.85,11.99,7.09,9.04,7.76,10.37,9.83,19.17,9.4,14.81


In [40]:

start = time.time()

names = list(ul_bx['n'])

current = [costs.count().sort_values(ascending = False).index[0]]
min_cost = costs[current].min(axis = 1).sum()
output = [[1, round(min_cost, 2), current[0]]]

for i in range(1, MAX_N_BOXES):
    min_name = ''
    for n in names:
        c = costs[current + [n]].min(axis = 1).sum()
        if c < min_cost:
            min_cost = c
            min_name = n
    if min_name != '':
        output.append([1 + i, round(min_cost, 2), min_name])
        current += [min_name]
    else:
        break
        
print(time.time() - start)

greedy_search = pd.DataFrame(output, columns = ['box_num', 'cumulative_cost', 'incremental_box'])
greedy_search = greedy_search.merge(ul_bx[['n', 'L', 'W', 'H', 'c']], left_on = 'incremental_box', right_on = 'n', how = 'inner')
greedy_search = greedy_search.drop(columns = ['n']).set_index('box_num')
greedy_search.to_csv(PATH + 'greedy_search.csv')
display(greedy_search)

36.00612211227417


,cumulative_cost,incremental_box,L,W,H,c
box_num,,,,,,
1,271482.59,S-4693,72.0,12.0,12.0,7.86
2,102135.65,S-4640,20.0,10.0,6.0,1.42
3,88409.39,S-4418,36.0,12.0,12.0,2.58
4,76652.99,S-4102,10.0,6.0,6.0,0.58
5,74850.53,S-4653,24.0,12.0,8.0,2.40
6,73097.90,S-4130,12.0,10.0,6.0,0.89
7,71846.82,S-4245,8.0,4.0,4.0,0.41
8,70710.60,S-19096,48.0,12.0,6.0,4.50
9,70133.58,S-4381,48.0,6.0,6.0,2.14


In [41]:
# cost comparison:
    # order volume:
o = es_df.copy()
o['v'] = o['l'] * o['w'] * o['h'] * o['q'] 
o = o[['o', 'v']].groupby('o', as_index = False).sum()
    # box cost:
b = ul_bx.copy()
b['V'] = b['L'] * b['W'] * b['H']
b = b[['n', 'c', 'V']]
    # shipping cost:
s = t[['Order No', 'Shipment Cost']].copy().rename(columns = {'Order No' : 'o'})

rows = []
current = []
for i in range(MAX_N_BOXES):
    current += [output[i][2]]
    df = costs[current].reset_index().melt(id_vars = ['o'], value_name = 'c_t')
    df = df[df['c_t'].notna()]
    df = df.sort_values('c_t').groupby('o', as_index = False).first()
    df = df.merge(o, on = 'o', how = 'inner').merge(b, on = 'n', how = 'inner').merge(s, on = 'o', how = 'inner')
    df['Sim Ship Cost'] = df['c_t'] - df['c']
    df = df[['v', 'V', 'Shipment Cost', 'Sim Ship Cost']].mean().apply(lambda x: round(x, 2))
    df['Fill Rate'] = str(round(df['v'] / df['V'] * 100,1))+'%'
    df['Incremental Box Name'] = output[i][2]
    df['Assortment Size'] = i + 1
    rows.append(df[['Assortment Size', 'Shipment Cost', 'Sim Ship Cost', 'Fill Rate', 'Incremental Box Name']])
    
rows = pd.concat(rows, axis = 1).transpose().set_index('Assortment Size')
rows.to_csv(PATH + 'greedy_search_comparison.csv')
display(rows)

,Shipment Cost,Sim Ship Cost,Fill Rate,Incremental Box Name
Assortment Size,,,,
1,8.57,27.7,5.2%,S-4693
2,8.57,10.95,20.9%,S-4640
3,8.57,9.65,27.4%,S-4418
4,8.57,8.55,37.3%,S-4102
5,8.57,8.31,40.5%,S-4653
6,8.57,8.18,42.9%,S-4130
7,8.57,8.05,44.8%,S-4245
8,8.57,7.89,48.3%,S-19096
9,8.57,7.84,50.0%,S-4381
